# Microsoft 365 Graph Tenant Ingestion

Extracts license and SKU intelligence from customer tenants **not captured in the Reliance Partner Center** registry,
using the same multi-tenant Entra app registration already used for Partner Center ingestion.

## Tables Produced

| Table | Grain | Represents |
|---|---|---|
| `graph_tenant_skus` | one row per tenant × SKU | License SKU inventory from Graph for non-PC tenants |
| `graph_tenant_promo_signals` | one row per tenant × promotion | Indicative promo match against active PC promotions |

## Prerequisites (One-Time in Azure Portal)

1. App registration → **Supported account types** → **"Accounts in any organizational directory (Multi-tenant)"**
2. App registration → **API permissions** → Add **Application** permissions:
   - `Microsoft Graph` → `CrossTenantInformation.ReadBasic.All` — resolves tenant ID → org name + domain
   - `Microsoft Graph` → `Organization.Read.All` — reads tenant profile
   - `Microsoft Graph` → `LicenseAssignment.Read.All` — reads subscribed SKUs
   - Click **Grant admin consent** for Reliance's tenant
3. For each non-PC customer: provide only their **Tenant ID** — cell 4 resolves the org name and domain automatically, then prints the consent URL to send to their Global Admin


In [13]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# Same app registration as PartnerCenterIngestion — must be set to multi-tenant
# and have Graph application permissions granted.

TENANT_ID     = "0b60fed4-5fc9-409d-95f2-271114f4c86f"   # Reliance tenant
CLIENT_ID     = "598e341b-9177-46fd-a104-df705cb8e036"   # Same app reg as PC ingestion
KEY_VAULT_URL = "https://dynamicsfabricsynckey.vault.azure.net/"
PC_SECRET_NAME = "Partnercenterxinforcerkey"              # Same client secret

GRAPH_BASE_URL  = "https://graph.microsoft.com/v1.0"
SCHEMA          = "dbo"

# ── Lakehouse write target ────────────────────────────────────────────────────
# Write directly to OneLake using ABFSS path — avoids needing a default
# lakehouse context in the Spark metastore.
WORKSPACE_ID    = "0f895a7e-09c6-4645-8b47-d272bc687b8a"
LAKEHOUSE_ID    = "3d0144b0-12bf-4483-9508-67b26b1fd125"   # ManagedServiceData
LAKEHOUSE_ABFSS = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
TABLES_PATH     = f"{LAKEHOUSE_ABFSS}/Tables/{SCHEMA}"

def lh_write(df, table_name):
    """Write a DataFrame as a Delta table to ManagedServiceData.dbo.<table_name>."""
    path = f"{TABLES_PATH}/{table_name}"
    (df.write.mode("overwrite").option("overwriteSchema", "true")
        .format("delta").save(path))
    return path

def lh_read(table_name):
    """Read a Delta table from ManagedServiceData.dbo.<table_name>."""
    return spark.read.format("delta").load(f"{TABLES_PATH}/{table_name}")

# ── Non-PC customers ────────────────────────────────────────────────────────
# Provide only tenant IDs — org name and domain are resolved automatically
# via the Microsoft Graph findTenantInformationByTenantId API.
CUSTOMER_TENANTS = [
    "c043e99a-5c7a-4aab-806e-2256c74620f3",
    # Add more tenant IDs here as needed
]

print(f"Configuration loaded. {len(CUSTOMER_TENANTS)} tenant(s) configured.")
print(f"  App       : {CLIENT_ID}")
print(f"  KV        : {KEY_VAULT_URL}")
print(f"  Tables at : {TABLES_PATH}")


StatementMeta(, 9768d946-1689-4be3-92f2-db324724454f, 4, Finished, Available, Finished, False)

Configuration loaded. 1 tenant(s) configured.
  App       : 598e341b-9177-46fd-a104-df705cb8e036
  KV        : https://dynamicsfabricsynckey.vault.azure.net/
  Tables at : abfss://0f895a7e-09c6-4645-8b47-d272bc687b8a@onelake.dfs.fabric.microsoft.com/3d0144b0-12bf-4483-9508-67b26b1fd125/Tables/dbo


In [2]:
# ============================================================================
# INSTALL DEPENDENCIES
# ============================================================================
%pip install msal --quiet


StatementMeta(, 05d62295-cece-484c-8b39-4c8df0104fdc, 13, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [14]:
# ============================================================================
# AUTHENTICATE — Microsoft Graph helpers
# ============================================================================
import msal, requests, json, time, re as _re
from datetime import datetime, timezone

print("Retrieving client secret from Key Vault...")
client_secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, PC_SECRET_NAME)

# Token cache: keyed by tenant_id so each customer gets its own cached token
_graph_token_cache = {}

def get_graph_token(tenant_id):
    """Return a valid app-only Graph Bearer token for the given tenant."""
    cache = _graph_token_cache.get(tenant_id, {"token": None, "expires_at": 0})
    if cache["token"] and time.time() < cache["expires_at"] - 60:
        return cache["token"]
    app = msal.ConfidentialClientApplication(
        client_id=CLIENT_ID,
        client_credential=client_secret,
        authority=f"https://login.microsoftonline.com/{tenant_id}",
    )
    result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
    if "access_token" not in result:
        raise RuntimeError(
            f"Graph token failed for {tenant_id}: {result.get('error')} — "
            f"{result.get('error_description')}"
        )
    cache = {"token": result["access_token"], "expires_at": time.time() + result.get("expires_in", 3600)}
    _graph_token_cache[tenant_id] = cache
    return cache["token"]

def graph_get(tenant_id, path, params=None):
    """App-only GET against Microsoft Graph in the specified tenant."""
    token = get_graph_token(tenant_id)
    url   = f"{GRAPH_BASE_URL}/{path.lstrip('/')}"
    resp  = requests.get(
        url,
        headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
        params=params,
        timeout=60,
    )
    if resp.status_code == 429:
        time.sleep(int(resp.headers.get("Retry-After", 10)))
        resp = requests.get(
            url,
            headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
            params=params,
            timeout=60,
        )
    resp.raise_for_status()
    return resp.json()

# ── Tenant info lookup ────────────────────────────────────────────────────────
_tenant_info_cache = {}

def resolve_tenant_info(customer_tenant_id):
    """Resolve a tenant ID → org display name + default domain via Graph API."""
    if customer_tenant_id in _tenant_info_cache:
        return _tenant_info_cache[customer_tenant_id]
    token = get_graph_token(TENANT_ID)
    url   = (
        f"{GRAPH_BASE_URL}/tenantRelationships/"
        f"findTenantInformationByTenantId(tenantId='{customer_tenant_id}')"
    )
    resp = requests.get(
        url,
        headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
        timeout=30,
    )
    resp.raise_for_status()
    info   = resp.json()
    result = {
        "tenant_id"     : info.get("tenantId", customer_tenant_id),
        "display_name"  : info.get("displayName", "Unknown"),
        "default_domain": info.get("defaultDomainName", ""),
    }
    _tenant_info_cache[customer_tenant_id] = result
    return result

# ── short_name helper (mirrors name_short() in build_semantic_layer) ─────────
_BIZ = (
    r'\s+(ltd|limited|plc|inc|incorporated|corp|corporation|llc|llp'
    r'|group|holdings|co|company|enterprises?|solutions?|services?'
    r'|technologies?|tech|consulting|international|global|africa'
    r'|nigeria|ghana|kenya)\.?$'
)

def py_short_name(name):
    """Strip common business-entity suffixes so LIKE matching works without them."""
    s = name.strip()
    s = _re.sub(_BIZ, '', s, flags=_re.IGNORECASE).strip()
    s = _re.sub(_BIZ, '', s, flags=_re.IGNORECASE).strip()
    return s.lower()

print("Auth helpers ready.")
print("  get_graph_token(tenant_id)     — app-only Graph token for any tenant")
print("  resolve_tenant_info(tenant_id) — resolves tenant ID → org name + domain")
print("  graph_get(tenant_id, path)     — Graph API call in a customer tenant")
print("  py_short_name(name)            — strips business suffixes for LIKE matching")
print()
print("Run cell 4 to look up tenant info and generate admin consent URLs.")


StatementMeta(, 9768d946-1689-4be3-92f2-db324724454f, 5, Finished, Available, Finished, False)

Retrieving client secret from Key Vault...
Auth helpers ready.
  get_graph_token(tenant_id)     — app-only Graph token for any tenant
  resolve_tenant_info(tenant_id) — resolves tenant ID → org name + domain
  graph_get(tenant_id, path)     — Graph API call in a customer tenant
  py_short_name(name)            — strips business suffixes for LIKE matching

Run cell 4 to look up tenant info and generate admin consent URLs.


In [ ]:
# ============================================================================
# TENANT LOOKUP + ADMIN CONSENT URL GENERATOR
# ============================================================================
# For each tenant ID in CUSTOMER_TENANTS, this cell:
#   1. Calls Microsoft Graph findTenantInformationByTenantId to resolve
#      the org display name and default domain — no customer consent needed
#   2. Builds the admin consent URL using the tenant ID directly
#   3. Prints a ready-to-send message for the customer's Global Admin
#
# Permissions the customer will be granting when they open the URL:
#   Organization.Read.All       — read tenant profile
#   LicenseAssignment.Read.All  — read subscribed SKUs and license counts

REDIRECT_URI = "https://localhost"

print("=" * 72)
print("TENANT LOOKUP + ADMIN CONSENT URLs")
print("=" * 72)
print()

resolved = []
for tid in CUSTOMER_TENANTS:
    try:
        info   = resolve_tenant_info(tid)
        name   = info["display_name"]
        domain = info["default_domain"]
        consent_url = (
            f"https://login.microsoftonline.com/{tid}/adminconsent"
            f"?client_id={CLIENT_ID}"
            f"&redirect_uri={REDIRECT_URI}"
            f"&state={tid}"
        )
        resolved.append({**info, "consent_url": consent_url})
        print(f"  Organisation : {name}")
        print(f"  Domain       : {domain}")
        print(f"  Tenant ID    : {tid}")
        print(f"  Consent URL  : {consent_url}")
        print()
    except Exception as e:
        print(f"  ✗ {tid} — lookup failed: {e}")
        print(f"    Ensure CrossTenantInformation.ReadBasic.All is granted on the app.")
        print()

print(f"Resolved {len(resolved)} / {len(CUSTOMER_TENANTS)} tenant(s).")
print()
print("Send each 'Consent URL' above to the customer's Microsoft 365 Global Admin.")
print("After they click Accept, this app can read their license data.")


StatementMeta(, 8afed6cd-cd54-452c-9935-e688c3e32232, 9, Finished, Available, Finished, False)

TENANT LOOKUP + ADMIN CONSENT URLs

  Organisation : Cloudware Limited
  Domain       : cloudware.africa
  Tenant ID    : c043e99a-5c7a-4aab-806e-2256c74620f3
  Consent URL  : https://login.microsoftonline.com/c043e99a-5c7a-4aab-806e-2256c74620f3/adminconsent?client_id=598e341b-9177-46fd-a104-df705cb8e036&redirect_uri=https://localhost&state=c043e99a-5c7a-4aab-806e-2256c74620f3

Resolved 1 / 1 tenant(s).

Send each 'Consent URL' above to the customer's Microsoft 365 Global Admin.
After they click Accept, this app can read their license data.


In [ ]:
# ============================================================================
# RESOLVED TENANT SUMMARY
# ============================================================================
# Shows the lookup results for all configured tenant IDs.
# Run this after cell 4 to confirm names and domains before proceeding.

print("Resolved tenant registry:\n")
print(f"  {'Tenant ID':<38} {'Organisation':<35} {'Default Domain'}")
print(f"  {'-'*38} {'-'*35} {'-'*35}")
for tid, info in _tenant_info_cache.items():
    print(f"  {tid:<38} {info['display_name']:<35} {info['default_domain']}")
print()
print(f"  {len(_tenant_info_cache)} tenant(s) resolved.")
print()
print("Proceeding to fetch subscribed SKUs from each consented tenant...")


StatementMeta(, 8afed6cd-cd54-452c-9935-e688c3e32232, 10, Finished, Available, Finished, False)

Resolved tenant registry:

  Tenant ID                              Organisation                        Default Domain
  -------------------------------------- ----------------------------------- -----------------------------------
  c043e99a-5c7a-4aab-806e-2256c74620f3   Cloudware Limited                   cloudware.africa

  1 tenant(s) resolved.

Proceeding to fetch subscribed SKUs from each consented tenant...


In [5]:
# ============================================================================
# STEP 1 — Fetch subscribed SKUs + subscription dates per tenant
# ============================================================================
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, LongType, BooleanType

sku_rows = []

for tenant_id in CUSTOMER_TENANTS:
    # Resolve org info from the cache populated by cell 4
    info         = _tenant_info_cache.get(tenant_id, {})
    company_name = info.get("display_name", "Unknown")
    domain       = info.get("default_domain", "")
    consent_status = "ok"

    try:
        # --- Subscription dates (one row per SKU, keyed by skuId) ---
        sub_dates = {}
        try:
            subs_data = graph_get(tenant_id, "/directory/subscriptions")
            for sub in subs_data.get("value", []):
                sid = sub.get("skuId")
                if sid and sid not in sub_dates:
                    sub_dates[sid] = {
                        "subscription_start": sub.get("createdDateTime"),
                        "subscription_end":   sub.get("nextLifecycleDateTime"),
                    }
        except Exception as e_sub:
            print(f"  WARN: could not fetch subscription dates for {company_name}: {e_sub}")

        # --- Subscribed SKUs (seat counts) ---
        data = graph_get(tenant_id, "/subscribedSkus")
        skus = data.get("value", [])
        print(f"  ✓ {company_name}: {len(skus)} SKU(s), {len(sub_dates)} subscription date(s)")

        for sku in skus:
            prepaid  = sku.get("prepaidUnits") or {}
            enabled  = prepaid.get("enabled", 0) or 0
            consumed = sku.get("consumedUnits", 0) or 0
            dates    = sub_dates.get(sku.get("skuId"), {})
            sku_rows.append({
                "tenant_id"             : tenant_id,
                "domain"                : domain,
                "company_name"          : company_name,
                "short_name"            : py_short_name(company_name),
                "sku_id"                : sku.get("skuId"),
                "sku_part_number"       : sku.get("skuPartNumber"),
                "capability_status"     : sku.get("capabilityStatus"),
                "consumed_units"        : consumed,
                "prepaid_enabled"       : enabled,
                "prepaid_suspended"     : prepaid.get("suspended", 0),
                "prepaid_warning"       : prepaid.get("warning", 0),
                "unassigned_seats"      : max(0, enabled - consumed),
                "enabled_service_plans" : json.dumps([
                    sp.get("servicePlanName")
                    for sp in sku.get("servicePlans", [])
                    if sp.get("capabilityStatus") == "Enabled"
                ]),
                "subscription_ids"      : json.dumps(sku.get("subscriptionIds", [])),
                "subscription_start"    : (dates.get("subscription_start") or "")[:10] or None,
                "subscription_end"      : (dates.get("subscription_end") or "")[:10] or None,
                "consent_status"        : consent_status,
                "ingested_at"           : datetime.now(timezone.utc).isoformat(),
            })

    except requests.HTTPError as e:
        status_code = e.response.status_code if e.response is not None else 0
        if status_code in (401, 403):
            consent_status = "consent_required"
            print(f"  ✗ {company_name} ({tenant_id}) [{status_code}] — admin consent not yet granted")
            sku_rows.append({
                "tenant_id": tenant_id, "domain": domain, "company_name": company_name,
                "short_name": py_short_name(company_name),
                "sku_id": None, "sku_part_number": None, "capability_status": None,
                "consumed_units": None, "prepaid_enabled": None, "prepaid_suspended": None,
                "prepaid_warning": None, "unassigned_seats": None,
                "enabled_service_plans": None, "subscription_ids": None,
                "subscription_start": None, "subscription_end": None,
                "consent_status": consent_status,
                "ingested_at": datetime.now(timezone.utc).isoformat(),
            })
        else:
            print(f"  ✗ {company_name} ({tenant_id}) [{status_code}] — {e}")
    except Exception as e:
        print(f"  ✗ {tenant_id} — {e}")

print(f"\nTotal SKU rows collected: {len(sku_rows)}")


StatementMeta(, 8afed6cd-cd54-452c-9935-e688c3e32232, 11, Finished, Available, Finished, False)

  ✓ Cloudware Limited: 20 SKU(s), 20 subscription date(s)

Total SKU rows collected: 20


In [7]:
# ============================================================================
# STEP 2 — Cross-reference with PC + write graph_tenant_skus
# ============================================================================
if not sku_rows:
    print("No SKU data collected — check consent status above.")
else:
    sku_pdf = pd.DataFrame(sku_rows)

    # Load PC customer registry to flag overlap
    try:
        pc_tenants    = lh_read("pc_customers").select("tenant_id").toPandas()
        pc_tenant_set = set(pc_tenants["tenant_id"].dropna().str.lower())
        sku_pdf["in_partner_center"] = sku_pdf["tenant_id"].str.lower().isin(pc_tenant_set)
    except Exception:
        print("  WARN: could not load pc_customers — in_partner_center will be False")
        sku_pdf["in_partner_center"] = False

    sku_sdf = spark.createDataFrame(sku_pdf)
    for c in ["sku_id", "sku_part_number", "capability_status", "enabled_service_plans",
              "subscription_ids", "consent_status", "tenant_id", "domain",
              "company_name", "short_name"]:
        sku_sdf = sku_sdf.withColumn(c, F.col(c).cast(StringType()))
    from pyspark.sql.types import DateType
    for c in ["subscription_start", "subscription_end"]:
        sku_sdf = sku_sdf.withColumn(c, F.col(c).cast(DateType()))
    for c in ["consumed_units", "prepaid_enabled", "prepaid_suspended",
              "prepaid_warning", "unassigned_seats"]:
        sku_sdf = sku_sdf.withColumn(c, F.col(c).cast(LongType()))
    sku_sdf = sku_sdf.withColumn("in_partner_center", F.col("in_partner_center").cast(BooleanType()))

    lh_write(sku_sdf, "graph_tenant_skus")
    print(f"Wrote dbo.graph_tenant_skus: {sku_sdf.count():,} rows")

    print("\nConsent status summary:")
    sku_sdf.groupBy("company_name", "consent_status", "in_partner_center").count().show(truncate=False)
    print("\nSample (short_name + SKUs + dates):")
    sku_sdf.filter(F.col("consent_status") == "ok").select(
        "short_name", "sku_part_number", "consumed_units", "prepaid_enabled",
        "unassigned_seats", "subscription_start", "subscription_end"
    ).show(20, truncate=False)


StatementMeta(, 8afed6cd-cd54-452c-9935-e688c3e32232, 13, Finished, Available, Finished, False)

Wrote dbo.graph_tenant_skus: 20 rows

Consent status summary:
+-----------------+--------------+-----------------+-----+
|company_name     |consent_status|in_partner_center|count|
+-----------------+--------------+-----------------+-----+
|Cloudware Limited|ok            |false            |20   |
+-----------------+--------------+-----------------+-----+


Sample (short_name + SKUs + dates):
+----------+---------------------------------------------------------------------+--------------+---------------+----------------+------------------+----------------+
|short_name|sku_part_number                                                      |consumed_units|prepaid_enabled|unassigned_seats|subscription_start|subscription_end|
+----------+---------------------------------------------------------------------+--------------+---------------+----------------+------------------+----------------+
|cloudware |VISIOCLIENT                                                          |3             |15     

In [ ]:
# ============================================================================
# STEP 3 — Preview: which PC promotions match this tenant's active SKUs?
# ============================================================================
# Queries pc_promotions directly — no separate table needed.
# The same promotions catalogue applies to all customers (PC and non-PC).
# This is a read-only preview; the agent queries the same tables live.

try:
    pc_promos  = lh_read("pc_promotions").select(
        F.col("sku_id").alias("promo_sku_id"),
        F.col("name").alias("promotion_name"),
        "discount_percentage", "end_date", "term_duration", "billing_cycle",
        "min_seats", "max_seats",
    )
    graph_skus = lh_read("graph_tenant_skus").filter(F.col("consent_status") == "ok")

    matches = (
        graph_skus
        .join(pc_promos,
              F.lower(graph_skus.sku_part_number) == F.lower(pc_promos.promo_sku_id),
              "inner")
        .select(
            graph_skus.short_name,
            graph_skus.company_name,
            graph_skus.sku_part_number,
            graph_skus.consumed_units,
            pc_promos.promotion_name,
            pc_promos.discount_percentage,
            pc_promos.end_date,
            pc_promos.min_seats,
            pc_promos.max_seats,
        )
        .orderBy("company_name", F.col("discount_percentage").desc())
    )

    cnt = matches.count()
    if cnt:
        print(f"Promo matches found: {cnt} row(s)")
        matches.show(30, truncate=False)
    else:
        print("No current PC promotions match any of this tenant's active SKUs.")

except Exception as e:
    print(f"Could not check promos: {e}")
    print("Ensure dbo.pc_promotions and dbo.graph_tenant_skus exist first.")


StatementMeta(, 2994e93e-6844-4e50-b9a4-a56ee96f7dcf, 44, Finished, Available, Finished, False)

Wrote dbo.graph_tenant_promo_signals: 0 rows
+---------+------+------------+---------------+--------------+---------------+------------+--------------+-------------------+--------+-------------+---------+---------+----------------+-----------+
|tenant_id|domain|company_name|sku_part_number|consumed_units|prepaid_enabled|promotion_id|promotion_name|discount_percentage|end_date|term_duration|min_seats|max_seats|eligibility_note|ingested_at|
+---------+------+------------+---------------+--------------+---------------+------------+--------------+-------------------+--------+-------------+---------+---------+----------------+-----------+
+---------+------+------------+---------------+--------------+---------------+------------+--------------+-------------------+--------+-------------+---------+---------+----------------+-----------+



In [ ]:
# ============================================================================
# SUMMARY
# ============================================================================
print("=" * 70)
print("M365 GRAPH TENANT INGESTION COMPLETE")
print("=" * 70)

for t in ["graph_tenant_skus", "graph_tenant_promo_signals"]:
    try:
        cnt = lh_read(t).count()
        print(f"  dbo.{t:<40s} {cnt:>6,} rows")
    except Exception:
        print(f"  dbo.{t:<40s} (not created)")

print()
print("Next steps:")
print("  1. Send admin consent URLs (cell 4) to any customer showing consent_required")
print("  2. Re-run cells 6-7 after each customer grants consent")
print("  3. Add consented non-PC tenants to build_semantic_layer to merge")
print("     graph_tenant_skus into sem_dim_customer via tenant_id join")


StatementMeta(, 2994e93e-6844-4e50-b9a4-a56ee96f7dcf, 45, Finished, Available, Finished, False)

M365 GRAPH TENANT INGESTION COMPLETE
  dbo.graph_tenant_skus                            20 rows
  dbo.graph_tenant_promo_signals                    0 rows

Next steps:
  1. Send admin consent URLs (cell 4) to any customer showing consent_required
  2. Re-run cells 6-7 after each customer grants consent
  3. Add consented non-PC tenants to build_semantic_layer to merge
     graph_tenant_skus into sem_dim_customer via tenant_id join


---

# Copilot Usage Report — Reliance Tenant

Fetches the Microsoft 365 Copilot usage report for **all licensed users on Reliance's own tenant** using the same app registration.

| Column | Description |
|---|---|
| `User Principal Name` | User's email / UPN |
| `Display Name` | Full name |
| `Last Activity Date` | Most recent Copilot activity across any app |
| `Microsoft Teams Copilot Last Activity Date` | Teams meeting & chat Copilot |
| `Word / Excel / PowerPoint / Outlook / OneNote / Loop Copilot Last Activity Date` | Per-app last use |
| `Copilot Chat Last Activity Date` | Microsoft 365 Copilot Chat (web/mobile) |
| `Report Period` | Days window (D7, D30, D90, D180) |

## Prerequisites (one-time)
- Azure Portal → App registrations → `598e341b...` → API permissions → Add **Application** permission: `Reports.Read.All` → Grant admin consent
- M365 Admin Center → Settings → Org settings → Reports → enable **"Show concealed user, group, and site names in reports"** (otherwise UPNs appear as GUIDs)

In [ ]:
import msal, requests, io, pandas as pd
from datetime import datetime

# ── Settings ─────────────────────────────────────────────────────────────────
PERIODS      = ["D30", "D90", "D180"]
COPILOT_BASE = "https://graph.microsoft.com/v1.0/copilot/reports"

# ── Force a fresh token (ensures new permissions are reflected) ───────────────
_app = msal.ConfidentialClientApplication(
    CLIENT_ID,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    client_credential=client_secret,
    token_cache=None
)
token = _app.acquire_token_for_client(
    scopes=["https://graph.microsoft.com/.default"]
)["access_token"]

def fetch_copilot_csv(token, period):
    """Fetches the Copilot usage CSV report and returns a DataFrame."""
    url = f"{COPILOT_BASE}/getMicrosoft365CopilotUsageUserDetail(period='{period}')?$format=text/csv"
    r1  = requests.get(url, headers={"Authorization": f"Bearer {token}"}, allow_redirects=False)
    if r1.status_code == 302:
        r2 = requests.get(r1.headers["Location"])
        r2.raise_for_status()
        return pd.read_csv(io.StringIO(r2.text))
    r1.raise_for_status()
    return pd.read_csv(io.StringIO(r1.text))

# ── Fetch, save, report ───────────────────────────────────────────────────────
copilot_reports = {}
date_str        = datetime.now().strftime("%Y%m%d")

for period in PERIODS:
    df = fetch_copilot_csv(token, period)
    copilot_reports[period] = df

    filename    = f"copilot_usage_{period}_{date_str}.csv"
    tmp_path    = f"/tmp/{filename}"
    dest_folder = f"{LAKEHOUSE_ABFSS}/Files/reports"

    df.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    notebookutils.fs.mkdirs(dest_folder)
    notebookutils.fs.cp(f"file:{tmp_path}", f"{dest_folder}/{filename}")

    active = df["Last Activity Date"].notna().sum()
    print(f"✅ {period}: {len(df)} users ({active} active) → Files/reports/{filename}")

print(f"\n⬇️  Download: Fabric portal → ManagedServiceData → Files → reports → right-click → Download")

StatementMeta(, 05d62295-cece-484c-8b39-4c8df0104fdc, 24, Finished, Available, Finished, False)

✅ D30: 80 users (79 active) → Files/reports/copilot_usage_D30_20260716.csv
✅ D90: 80 users (79 active) → Files/reports/copilot_usage_D90_20260716.csv
✅ D180: 80 users (79 active) → Files/reports/copilot_usage_D180_20260716.csv

⬇️  Download: Fabric portal → ManagedServiceData → Files → reports → right-click → Download


In [10]:
import msal, requests, json, io, pandas as pd

# Force a brand-new token (bypasses MSAL in-memory cache)
_fresh_app = msal.ConfidentialClientApplication(
    CLIENT_ID,
    authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    client_credential=client_secret,
    token_cache=None
)
_result = _fresh_app.acquire_token_for_client(
    scopes=["https://graph.microsoft.com/.default"]
)
fresh_token = _result["access_token"]

# Try CSV format on the new /copilot/reports/ path
test_url = (
    "https://graph.microsoft.com/v1.0/copilot/reports/"
    "getMicrosoft365CopilotUsageUserDetail(period='D7')?$format=text/csv"
)
r = requests.get(
    test_url,
    headers={"Authorization": f"Bearer {fresh_token}"},
    allow_redirects=False   # capture 302 redirect if present
)
print(f"Status : {r.status_code}")

if r.status_code == 302:
    r2 = requests.get(r.headers["Location"])
    print(f"Redirect status: {r2.status_code}")
    df_test = pd.read_csv(io.StringIO(r2.text))
    print(f"✅ {len(df_test)} rows, columns: {list(df_test.columns)}")
elif r.status_code == 200:
    df_test = pd.read_csv(io.StringIO(r.text))
    print(f"✅ {len(df_test)} rows, columns: {list(df_test.columns)}")
else:
    print(json.dumps(r.json(), indent=2))

StatementMeta(, 05d62295-cece-484c-8b39-4c8df0104fdc, 23, Finished, Available, Finished, False)

Status : 302
Redirect status: 200
✅ 80 rows, columns: ['Report Refresh Date', 'User Principal Name', 'Display Name', 'Last Activity Date', 'Copilot Chat Last Activity Date', 'Microsoft Teams Copilot Last Activity Date', 'Word Copilot Last Activity Date', 'Excel Copilot Last Activity Date', 'PowerPoint Copilot Last Activity Date', 'Outlook Copilot Last Activity Date', 'OneNote Copilot Last Activity Date', 'Loop Copilot Last Activity Date', 'Report Period']


In [ ]:
# ── Preview each period ───────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)

for period, df in copilot_reports.items():
    print(f"\n{'='*60}")
    print(f"  Period: {period}  |  {len(df)} users  |  {len(df.columns)} columns")
    print(f"{'='*60}")
    display(df)

In [16]:
# ============================================================================
# LAYER 2 — User Count Summary + Daily Trend (D90)
# Writes: dbo.copilot_user_count_summary, dbo.copilot_user_count_trend
# ============================================================================
import io, requests, pandas as pd, msal, re
from datetime import datetime

COPILOT_BASE = "https://graph.microsoft.com/v1.0/copilot/reports"

# Fresh token
_app = msal.ConfidentialClientApplication(
    CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    client_credential=client_secret, token_cache=None
)
token = _app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])["access_token"]

def fetch_csv(url):
    """Fetch any Graph CSV report — handles 302 redirect."""
    r = requests.get(url, headers={"Authorization": f"Bearer {token}"}, allow_redirects=False)
    if r.status_code == 302:
        r2 = requests.get(r.headers["Location"])
        r2.raise_for_status()
        return pd.read_csv(io.StringIO(r2.text))
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def clean_cols(df):
    """Sanitize column names for Delta: lowercase, spaces/special chars → underscore."""
    df.columns = [re.sub(r'[^a-zA-Z0-9]', '_', c).lower().strip('_') for c in df.columns]
    return df

date_str    = datetime.now().strftime("%Y%m%d")
dest_folder = f"{LAKEHOUSE_ABFSS}/Files/reports"

# ── Summary (one row: enabled vs active counts per app) ──────────────────────
df_summary = clean_cols(fetch_csv(f"{COPILOT_BASE}/getMicrosoft365CopilotUserCountSummary(period='D90')?$format=text/csv"))
print(f"Summary  : {len(df_summary)} rows, cols={list(df_summary.columns)}")

# ── Trend (one row per day × app) ────────────────────────────────────────────
df_trend = clean_cols(fetch_csv(f"{COPILOT_BASE}/getMicrosoft365CopilotUserCountTrend(period='D90')?$format=text/csv"))
print(f"Trend    : {len(df_trend)} rows (daily)")

# ── Save CSV ──────────────────────────────────────────────────────────────────
for name, df in [("copilot_summary_D90", df_summary), ("copilot_trend_D90", df_trend)]:
    fn = f"{name}_{date_str}.csv"
    df.to_csv(f"/tmp/{fn}", index=False, encoding="utf-8-sig")
    notebookutils.fs.cp(f"file:/tmp/{fn}", f"{dest_folder}/{fn}")

# ── Save Delta tables ─────────────────────────────────────────────────────────
spark.createDataFrame(df_summary).write.mode("overwrite").option("overwriteSchema","true").format("delta").save(f"{TABLES_PATH}/copilot_user_count_summary")
spark.createDataFrame(df_trend).write.mode("overwrite").option("overwriteSchema","true").format("delta").save(f"{TABLES_PATH}/copilot_user_count_trend")

print("\n✅ dbo.copilot_user_count_summary saved")
print("✅ dbo.copilot_user_count_trend saved")

display(df_summary)
display(df_trend.head(10))

StatementMeta(, 9768d946-1689-4be3-92f2-db324724454f, 7, Finished, Available, Finished, False)

Summary  : 1 rows, cols=['report_refresh_date', 'report_period', 'microsoft_teams_enabled_users', 'microsoft_teams_active_users', 'word_enabled_users', 'word_active_users', 'powerpoint_enabled_users', 'powerpoint_active_users', 'outlook_enabled_users', 'outlook_active_users', 'excel_enabled_users', 'excel_active_users', 'onenote_enabled_users', 'onenote_active_users', 'loop_enabled_users', 'loop_active_users', 'any_app_enabled_users', 'any_app_active_users', 'copilot_chat_enabled_users', 'copilot_chat_active_users']
Trend    : 90 rows (daily)

✅ dbo.copilot_user_count_summary saved
✅ dbo.copilot_user_count_trend saved


SynapseWidget(Synapse.DataFrame, e83ccee8-d012-4b5b-810b-bb79d8cbcaf8)

SynapseWidget(Synapse.DataFrame, 3fe3baa3-d880-4ba5-b04f-7d15c5bc10cb)

In [22]:
# ============================================================================
# LAYER 3 — Entra ID User Profiles
# ============================================================================
import requests, pandas as pd, re, json, msal

GRAPH_V1_URL = "https://graph.microsoft.com/v1.0"

# Fresh token — picks up newly granted User.Read.All
_app  = msal.ConfidentialClientApplication(CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}", client_credential=client_secret, token_cache=None)
token = _app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])["access_token"]

SELECT = "id,displayName,userPrincipalName,department,jobTitle,country,officeLocation,city,companyName"
url    = f"{GRAPH_V1_URL}/users?$select={SELECT}&$top=999"

users = []
while url:
    r = requests.get(url, headers={"Authorization": f"Bearer {token}"})
    if r.status_code != 200:
        print(f"❌ {r.status_code}: {json.dumps(r.json(), indent=2)}")
        break
    data  = r.json()
    users.extend(data.get("value", []))
    url   = data.get("@odata.nextLink")

if users:
    df_users = pd.DataFrame(users)
    df_users.columns = [re.sub(r'[^a-zA-Z0-9]', '_', c).lower().strip('_') for c in df_users.columns]
    print(f"✅ {len(df_users)} users, cols: {list(df_users.columns)}")

    fn = f"entra_user_profiles_{date_str}.csv"
    df_users.to_csv(f"/tmp/{fn}", index=False, encoding="utf-8-sig")
    notebookutils.fs.cp(f"file:/tmp/{fn}", f"{dest_folder}/{fn}")
    spark.createDataFrame(df_users).write.mode("overwrite").option("overwriteSchema","true").format("delta").save(f"{TABLES_PATH}/entra_user_profiles")
    print("✅ dbo.entra_user_profiles saved")
    display(df_users[df_users["department"].notna()].head(15))

StatementMeta(, 9768d946-1689-4be3-92f2-db324724454f, 14, Finished, Available, Finished, False)

✅ 1538 users, cols: ['id', 'displayname', 'userprincipalname', 'department', 'jobtitle', 'country', 'officelocation', 'city', 'companyname', 'odata_type']
✅ dbo.entra_user_profiles saved


SynapseWidget(Synapse.DataFrame, 33dc83d1-9c6c-42c2-8c16-8303d929dc01)

In [ ]:
# ============================================================================
# LAYER 4 — AI Interaction History (Prompts, Responses, Intent, App)
# ============================================================================
import requests, pandas as pd, re, json, msal
from datetime import datetime, timedelta, timezone

# Fresh token — picks up newly granted AiEnterpriseInteraction.Read.All
_app  = msal.ConfidentialClientApplication(CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}", client_credential=client_secret, token_cache=None)
token = _app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])["access_token"]

end_dt   = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=90)
fmt      = "%Y-%m-%dT%H:%M:%SZ"

url = (
    f"{GRAPH_V1_URL}/copilot/interactionHistory/getAllEnterpriseInteractions"
    f"?$filter=createdDateTime ge {start_dt.strftime(fmt)} "
    f"and createdDateTime le {end_dt.strftime(fmt)}"
    f"&$top=50"
)

r = requests.get(url, headers={"Authorization": f"Bearer {token}"})
print(f"Status: {r.status_code}")

if r.status_code in (403, 404):
    print(json.dumps(r.json(), indent=2))
else:
    r.raise_for_status()
    interactions, page = [], 0
    data = r.json()
    while data and page < 200:
        for item in data.get("value", []):
            user_msg = next((m for m in item.get("interactions", []) if m.get("from", {}).get("role") == "user"), {})
            ai_msg   = next((m for m in item.get("interactions", []) if m.get("from", {}).get("role") == "assistant"), {})
            interactions.append({
                "interaction_id":     item.get("id"),
                "created_datetime":   item.get("createdDateTime"),
                "app_class":          item.get("appClass"),
                "copilot_experience": item.get("copilotExperience"),
                "user_id":            item.get("from", {}).get("user", {}).get("id"),
                "prompt":             user_msg.get("body", {}).get("content", ""),
                "response":           ai_msg.get("body", {}).get("content", ""),
                "context_resources":  len(item.get("contexts", [])),
            })
        next_url = data.get("@odata.nextLink")
        if not next_url:
            break
        r2   = requests.get(next_url, headers={"Authorization": f"Bearer {token}"})
        r2.raise_for_status()
        data = r2.json()
        page += 1

    if interactions:
        df_int = pd.DataFrame(interactions)
        df_int.columns = [re.sub(r'[^a-zA-Z0-9]', '_', c).lower().strip('_') for c in df_int.columns]
        fn = f"copilot_interactions_90d_{date_str}.csv"
        df_int.to_csv(f"/tmp/{fn}", index=False, encoding="utf-8-sig")
        notebookutils.fs.cp(f"file:/tmp/{fn}", f"{dest_folder}/{fn}")
        spark.createDataFrame(df_int).write.mode("overwrite").option("overwriteSchema","true").format("delta").save(f"{TABLES_PATH}/copilot_interactions")
        print(f"✅ {len(df_int)} interactions → dbo.copilot_interactions")
        display(df_int.head(10))
    else:
        print("ℹ️  No interactions found in the last 90 days.")

StatementMeta(, 9768d946-1689-4be3-92f2-db324724454f, 16, Finished, Available, Finished, False)

Status: 404
{
  "error": {
    "code": "NotFound",
    "message": "Requested API is not supported. Please check the path.",
    "innerError": {
      "date": "2026-07-16T16:32:06",
      "request-id": "ae595272-f7de-4936-8326-10d01369cf64",
      "client-request-id": "ae595272-f7de-4936-8326-10d01369cf64"
    }
  }
}
